# Text Methods

A normal Python string has a variety of method calls available:

In [ ]:
mystring = 'hello'

In [ ]:
mystring.capitalize()

'Hello'

In [ ]:
mystring.isdigit()

False

In [ ]:
help(str)

Help on class str in module builtins:

class str(object)
 |  str(object='') -> str
 |  str(bytes_or_buffer[, encoding[, errors]]) -> str
 |  
 |  Create a new string object from the given object. If encoding or
 |  errors is specified, then the object must expose a data buffer
 |  that will be decoded using the given encoding and error handler.
 |  Otherwise, returns the result of object.__str__() (if defined)
 |  or repr(object).
 |  encoding defaults to sys.getdefaultencoding().
 |  errors defaults to 'strict'.
 |  
 |  Methods defined here:
 |  
 |  __add__(self, value, /)
 |      Return self+value.
 |  
 |  __contains__(self, key, /)
 |      Return key in self.
 |  
 |  __eq__(self, value, /)
 |      Return self==value.
 |  
 |  __format__(self, format_spec, /)
 |      Return a formatted version of the string as described by format_spec.
 |  
 |  __ge__(self, value, /)
 |      Return self>=value.
 |  
 |  __getattribute__(self, name, /)
 |      Return getattr(self, name).
 |  
 |  

# Pandas and Text

Pandas can do a lot more than what we show here. Full online documentation on things like advanced string indexing and regular expressions with pandas can be found here: https://pandas.pydata.org/docs/user_guide/text.html

## Text Methods on Pandas String Column

In [ ]:
import pandas as pd

In [ ]:
names = pd.Series(['andrew','bobo','claire','david','4'])

In [ ]:
names

0    andrew
1      bobo
2    claire
3     david
4         4
dtype: object

In [ ]:
names.str.capitalize()

0    Andrew
1      Bobo
2    Claire
3     David
4         4
dtype: object

In [ ]:
names.str.isdigit()

0    False
1    False
2    False
3    False
4     True
dtype: bool

## Splitting , Grabbing, and Expanding

In [ ]:
tech_finance = ['GOOG,APPL,AMZN','JPM,BAC,GS']

In [ ]:
len(tech_finance)

2

In [ ]:
tickers = pd.Series(tech_finance)

In [ ]:
tickers

0    GOOG,APPL,AMZN
1        JPM,BAC,GS
dtype: object

In [ ]:
tickers.str.split(',')

0    [GOOG, APPL, AMZN]
1        [JPM, BAC, GS]
dtype: object

In [ ]:
tickers.str.split(',').str[0]

0    GOOG
1     JPM
dtype: object

In [ ]:
tickers.str.split(',',expand=True)

,0,1,2
0,GOOG,APPL,AMZN
1,JPM,BAC,GS


## Cleaning or Editing Strings

In [ ]:
messy_names = pd.Series(["andrew  ","bo;bo","  claire  "])

In [ ]:
# Notice the "mis-alignment" on the right hand side due to spacing in "andrew  " and "  claire  "
messy_names

0      andrew  
1         bo;bo
2      claire  
dtype: object

In [ ]:
messy_names.str.replace(";","")

0      andrew  
1          bobo
2      claire  
dtype: object

In [ ]:
messy_names.str.strip()

0    andrew
1     bo;bo
2    claire
dtype: object

In [ ]:
messy_names.str.replace(";","").str.strip()

0    andrew
1      bobo
2    claire
dtype: object

In [ ]:
messy_names.str.replace(";","").str.strip().str.capitalize()

0    Andrew
1      Bobo
2    Claire
dtype: object

## Alternative with Custom apply() call

In [ ]:
def cleanup(name):
    name = name.replace(";","")
    name = name.strip()
    name = name.capitalize()
    return name

In [ ]:
messy_names

0      andrew  
1         bo;bo
2      claire  
dtype: object

In [ ]:
messy_names.apply(cleanup)

0    Andrew
1      Bobo
2    Claire
dtype: object

## Which one is more efficient?

In [ ]:
import timeit

# code snippet to be executed only once
setup = '''
import pandas as pd
import numpy as np
messy_names = pd.Series(["andrew  ","bo;bo","  claire  "])
def cleanup(name):
    name = name.replace(";","")
    name = name.strip()
    name = name.capitalize()
    return name
'''

# code snippet whose execution time is to be measured
stmt_pandas_str = '''
messy_names.str.replace(";","").str.strip().str.capitalize()
'''

stmt_pandas_apply = '''
messy_names.apply(cleanup)
'''

stmt_pandas_vectorize='''
np.vectorize(cleanup)(messy_names)
'''

In [ ]:
timeit.timeit(setup = setup,
                    stmt = stmt_pandas_str,
                    number = 10000)

3.931618999999955

In [ ]:
timeit.timeit(setup = setup,
                    stmt = stmt_pandas_apply,
                    number = 10000)

1.2268500999999787

In [ ]:
timeit.timeit(setup = setup,
                    stmt = stmt_pandas_vectorize,
                    number = 10000)

0.28283379999993485

Wow! While .str() methods can be extremely convienent, when it comes to performance, don't forget about np.vectorize()! Review the "Useful Methods" lecture for a deeper discussion on np.vectorize()